# Simple EDA: Current AmazonQAC sample

This notebook explains what the current data represents, its shape, columns, types, missing values, uniqueness, text sizes, time coverage, and popularity. It is deliberately small and read-only.

**Current project stage:** M1 — environment and schema. This notebook explores the saved raw sample; it does not normalize, split, train, or modify the data.

## 1. Load the data

Each row is one Amazon search interaction. It contains a sequence of text typed by a shopper, the final search submitted, event times, identifiers, and Amazon-supplied popularity metadata.

In [ ]:
from pathlib import Path

import pandas as pd
from pyarrow import parquet

# This works whether the notebook starts in project/ or its parent folder.
possible_paths = [
    Path("datasets/amazonqac/raw/train_sample_20k.parquet"),
    Path("project/datasets/amazonqac/raw/train_sample_20k.parquet"),
]
data_path = next((path for path in possible_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find train_sample_20k.parquet")

df = pd.read_parquet(data_path)
print(f"Loaded: {data_path.resolve()}")
print(f"File size: {data_path.stat().st_size / (1024**2):.2f} MiB")

## 2. Shape, columns, and physical types

The current sample contains **20,000 rows and 7 columns**. The `prefixes` column is important: each cell contains a list of typing states, not one ordinary string.

In [ ]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())
print("\nParquet schema:")
print(parquet.read_schema(data_path))

### What each column means

| Column | Meaning | Project role |
|---|---|---|
| `query_id` | Unique interaction ID | Identity and duplicate checks; not a model feature |
| `session_id` | Groups interactions from one session | Diagnostics; excluded from the first model |
| `prefixes` | Sequence of observed typing states | Understand input behaviour and create development requests |
| `first_prefix_typed_time` | When typing began | Chronological split and timing checks |
| `final_search_term` | Search eventually submitted | Training text and evaluation target |
| `search_time` | When the final search was submitted | Chronology and timing checks |
| `popularity` | Amazon-supplied historical count | Optional metadata; not frequency inside this 20K sample |

The main relationship is: **typing history (`prefixes`) → final submitted search (`final_search_term`)**.

## 3. Readable examples

The opaque session ID is deliberately excluded from this display.

In [ ]:
examples = df.head(5).copy()
examples["number_of_prefixes"] = examples["prefixes"].map(len)
examples["first_recorded_prefix"] = examples["prefixes"].map(lambda values: values[0])
examples["last_recorded_prefix"] = examples["prefixes"].map(lambda values: values[-1])
examples[
    [
        "query_id",
        "first_recorded_prefix",
        "last_recorded_prefix",
        "number_of_prefixes",
        "final_search_term",
        "popularity",
    ]
]

## 4. Missing values, blanks, and uniqueness

The current sample has no null cells, no blank text values, and no duplicate query IDs. Repeated final searches are expected because different interactions can submit the same search.

In [ ]:
text_columns = [
    "session_id",
    "first_prefix_typed_time",
    "final_search_term",
    "search_time",
]
blank_counts = {
    column: int(df[column].str.strip().eq("").sum()) for column in text_columns
}
blank_prefixes = sum(
    str(prefix).strip() == "" for history in df["prefixes"] for prefix in history
)
unique_counts = {
    column: ("nested lists" if column == "prefixes" else df[column].nunique())
    for column in df.columns
}
quality = pd.DataFrame(
    {
        "nulls": df.isna().sum(),
        "unique values": pd.Series(unique_counts),
    }
)
display(quality)
print("Blank text values:", blank_counts)
print("Blank values inside prefix histories:", blank_prefixes)
print("Duplicate query IDs:", int(df["query_id"].duplicated().sum()))
print("Repeated final-search rows:", int(df["final_search_term"].duplicated().sum()))
print("Unique session IDs:", df["session_id"].nunique())

## 5. Text and typing-history sizes

The typical final search is 19 characters long. An interaction records about 10 typing states on average, but histories range from 1 to 81 states.

In [ ]:
term_lengths = df["final_search_term"].str.len()
prefix_counts = df["prefixes"].map(len)
all_prefixes = pd.Series(
    (prefix for history in df["prefixes"] for prefix in history),
    dtype="string",
)
prefix_lengths = all_prefixes.str.len()

size_summary = pd.DataFrame(
    {
        "final-search length": term_lengths.describe(),
        "typing states per row": prefix_counts.describe(),
        "typing-state length": prefix_lengths.describe(),
    }
)
display(size_summary.round(2))
print(f"Total recorded typing states: {len(all_prefixes):,}")

## 6. Typing histories include edits and rewrites

A history is not always a clean sequence of longer prefixes. Shoppers may delete text, correct it, or change the intended search. This matters because a pure autocomplete model can extend a prefix but cannot repair every rewrite.

In [ ]:
rows_with_any_compatible_prefix = sum(
    any(term.startswith(str(prefix)) for prefix in history)
    for history, term in zip(df["prefixes"], df["final_search_term"], strict=True)
)
rows_with_all_compatible_prefixes = sum(
    all(term.startswith(str(prefix)) for prefix in history)
    for history, term in zip(df["prefixes"], df["final_search_term"], strict=True)
)
print(
    f"At least one literal prefix: {rows_with_any_compatible_prefix:,} "
    f"({rows_with_any_compatible_prefix / len(df):.1%})"
)
print(
    f"Every state is a literal prefix: {rows_with_all_compatible_prefixes:,} "
    f"({rows_with_all_compatible_prefixes / len(df):.1%})"
)

## 7. Time coverage

The sample covers all of September 2023. The timestamp strings use mixed fractional-second precision, so `format="mixed"` is intentional. Tiny negative durations can result from the final-search time being stored only to whole seconds. Very large durations also exist and should be investigated before elapsed time is used as a feature.

In [ ]:
first_time = pd.to_datetime(
    df["first_prefix_typed_time"], format="mixed", errors="coerce", utc=True
)
search_time = pd.to_datetime(
    df["search_time"], format="mixed", errors="coerce", utc=True
)
elapsed_seconds = (search_time - first_time).dt.total_seconds()
cutoff = pd.Timestamp("2023-09-25 00:00:00", tz="UTC")

print("First search:", search_time.min())
print("Last search: ", search_time.max())
print("Unparseable first-prefix times:", int(first_time.isna().sum()))
print("Unparseable search times:", int(search_time.isna().sum()))
print("Rows before 25 September:", int((search_time < cutoff).sum()))
print("Rows at/after 25 September:", int((search_time >= cutoff).sum()))
display(
    elapsed_seconds.describe(percentiles=[0.5, 0.95, 0.99]).round(3).to_frame("seconds")
)

## 7a. Interaction volume across time

This is pre-model EDA. It shows whether the sample has enough earlier and later interactions for a chronological split. Times stay in their source convention; this cell does not apply a timezone conversion.

In [ ]:
first_time_source = pd.to_datetime(
    df["first_prefix_typed_time"], format="mixed", errors="coerce"
)
search_time_source = pd.to_datetime(df["search_time"], format="mixed", errors="coerce")

daily_interactions = (
    search_time_source.dt.floor("D")
    .value_counts()
    .sort_index()
    .rename_axis("date")
    .reset_index(name="interactions")
)
display(daily_interactions)

daily_interactions.plot(
    x="date",
    y="interactions",
    kind="bar",
    figsize=(12, 4),
    title="Sample interactions by final-search date",
    legend=False,
)

## 7b. Candidate cutoff coverage

Compare several cutoffs using only time coverage and data quality—before fitting or scoring any model. A record is assigned only when both timestamps lie on the same side of the cutoff.

In [ ]:
candidate_cutoffs = pd.to_datetime(
    [
        "2023-09-20",
        "2023-09-22",
        "2023-09-24",
        "2023-09-25",
        "2023-09-26",
        "2023-09-28",
    ]
)

parseable_times = first_time_source.notna() & search_time_source.notna()
reversed_times = parseable_times & (search_time_source < first_time_source)

cutoff_rows = []
for candidate in candidate_cutoffs:
    crosses_cutoff = (
        parseable_times
        & ~reversed_times
        & (first_time_source < candidate)
        & (search_time_source >= candidate)
    )
    usable = parseable_times & ~reversed_times & ~crosses_cutoff
    train = usable & (search_time_source < candidate)
    development = usable & (search_time_source >= candidate)
    cutoff_rows.append(
        {
            "cutoff": candidate.date(),
            "train": int(train.sum()),
            "development": int(development.sum()),
            "crosses cutoff": int(crosses_cutoff.sum()),
            "reversed time": int(reversed_times.sum()),
            "unparseable time": int((~parseable_times).sum()),
        }
    )

cutoff_coverage = pd.DataFrame(cutoff_rows)
cutoff_coverage["development share"] = cutoff_coverage["development"] / (
    cutoff_coverage["train"] + cutoff_coverage["development"]
)
display(cutoff_coverage)

## 7c. Working-cutoff data-quality diagnostics

The table below explains what is kept by the current working cutoff (25 September) and what is quarantined. Quarantined records remain available for inspection but are excluded from both training and development.

In [ ]:
WORKING_CUTOFF = pd.Timestamp("2023-09-25 00:00:00")
crosses_cutoff = (
    parseable_times
    & ~reversed_times
    & (first_time_source < WORKING_CUTOFF)
    & (search_time_source >= WORKING_CUTOFF)
)
usable = parseable_times & ~reversed_times & ~crosses_cutoff

partition_summary = pd.Series(
    {
        "training (both times before cutoff)": int(
            (usable & (search_time_source < WORKING_CUTOFF)).sum()
        ),
        "development (both times at/after cutoff)": int(
            (usable & (search_time_source >= WORKING_CUTOFF)).sum()
        ),
        "quarantine: crosses cutoff": int(crosses_cutoff.sum()),
        "quarantine: reversed time": int(reversed_times.sum()),
        "quarantine: unparseable time": int((~parseable_times).sum()),
    },
    name="records",
)

display(partition_summary.to_frame())
assert partition_summary.sum() == len(df)

## 8. Popularity and repeated searches

The supplied `popularity` field is strongly skewed: its median is 62, while its mean is about 2,556 and its maximum is 179,976. It describes wider Amazon history and is **not** the number of occurrences in this selected sample. The two tables below deliberately keep those measures separate.

In [ ]:
display(
    df["popularity"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2)
)

print("Most frequent final searches inside this 20K sample:")
display(df["final_search_term"].value_counts().head(10).to_frame("sample rows"))

print("Highest Amazon-supplied popularity values:")
display(
    df.groupby("final_search_term")["popularity"]
    .max()
    .nlargest(10)
    .to_frame("supplied popularity")
)

## Main takeaways

1. This is **search interaction data**, not a product catalogue.
2. `final_search_term` is the submitted search and the main training/evaluation text.
3. `prefixes` is a nested typing history that needs special handling.
4. IDs and times are for validation, splitting, and diagnostics—not v1 prediction features.
5. Supplied popularity must remain separate from train-partition occurrence counts.
6. Rewrite behaviour and timing outliers should be preserved and investigated, not automatically removed.

**Stopping point:** this notebook answers the basic questions about the current raw data. Normalization, partitioning, feature engineering, and modeling belong to later project stages.